# Dataset Preparation for MR-LFADS

This tutorial explains how to format your dataset for use with the MR-LFADS datamodule. It covers:

1. The standard data structure and conventions for the `BasicDataModule`
2. [**Advanced**] How to implement a custom datamodule for specialized use cases

## Data Structure

The `mrlfads.datamodules.BasicDataModule` is the standard datamodule for MR-LFADS. It expects an HDF5 file (`data.h5`) with the following structure:

```text
data.h5
├── session_index/
│   ├── area-<brain_area_name>      # neural activity (type: "hidden_state")
│   ├── inputs-<brain_area_name>    # inputs to a brain area (type: "inputs")
│   └── info                        # metadata (type: "info")
```

Each `session_index` group corresponds to a single recording session.

## Dataset Requirements

Within each `session_index`, datasets must follow these conventions:

### Neural Activity
- **Naming**: area-`<brain_area_name>`
- **Type**: `"hidden_state"`
- **Shape**: `(n_trials, n_time, n_neurons)`
- **Description**: Neural activity for a specific brain area `<brain_area_name>`
    
### Inputs
- **Naming**: inputs-`<brain_area_name>`
- **Type**: `"inputs"`
- **Shape**: `(n_trials, n_time, n_input_features)`
- **Description**: External inputs associated with a specific brain area `<brain_area_name>`

### Metadata (`info`)
- **Naming**: arbitrary 
- **Type**: `"info"`  
- **Shape**: `(n_trials, n_time, n_info_dims)`  
- **Description**: Session-level metadata aligned to each trial and time step  

## Test

To verify that your dataset loads correctly, specify the data file and brain area names:

In [ ]:
filename = NotImplemented  # path relative to config.paths.datapath
area_names = NotImplemented  # list of area names, e.g. ["Area1", "Area2"]

Then initialize and setup the datamodule:

In [ ]:
from mrlfads.datamodules import BasicDataModule

dm = BasicDataModule(
    filename=filename,
    area_names=area_names,
    session_idxs=[0],
)
dm.setup()

### Notes

* `filename` should point to your data.h5 file relative to config.paths.datapath
* `area_names` must match the suffixes used in your dataset (e.g., area-M1 → "M1")
* `session_idxs=[0]` loads the first session; adjust if multiple sessions are present

## [**Advanced**] Custom Datamodules

The `mrlfads.model.MRLFADS` model expects data in the following format for each `session_index`:

* A `mrlfads.utils.common_utils.Batch` object (named tuple) containing:
    - neural data
    - external inputs
* A dictionary of metadata, where each entry is aligned per trial

To include custom metadata (e.g., for evaluation or analysis), you can modify the `setup()` method of your datamodule. The key requirement is that all data must be formatted correctly before being passed to `mrlfads.datamodules.SessionAreaDataset`:

In [ ]:
# Do not execute this cell---locate this line in mrlfads.datamodules.py for reference

session_dataset = SessionAreaDataset(
    area_data_dict,
    info_strings,
    ext_input_dict,
)

`info_strings` contains the metadata and should be structured as a list of dictionaries, where:
* Each element corresponds to a single trial
* Each dictionary contains metadata for that trial
* The structure is as follows:

In [ ]:
# Do not execute this cell

info_strings = [
    {
        "metadata_name": np.array(...),  # any shape, but aligned to the trial
        ...
    },
    ...
]